In [8]:
import json
import os
from pathlib import Path

import pandas as pd

In [9]:
top_path = Path(os.path.dirname(os.getcwd()))
data_path = top_path / "data"
insights_path = top_path / "reports" / "insights"

notebooks_path = top_path / "notebooks"
team_data_path = data_path / "processed" / "team_data.parquet"
predictions_path = insights_path / "OutcomePrediction_predictions.parquet"

In [10]:
predictions_df = pd.read_parquet(predictions_path, engine="pyarrow").rename(
    columns={"result": "prediction"}
)
team_df = pd.read_parquet(team_data_path, engine="pyarrow")

## Store the accuracy per league in a dictionary

In [11]:
league_data = team_df[["gameid", "side", "league", "result"]]
predictions_df = predictions_df.merge(
    league_data, on=["gameid", "side"], how="inner", validate="many_to_many"
)

In [12]:
# Explore accuracy by league
leagues = predictions_df["league"].unique()
accuracies = {}

for league in leagues:
    league_df = predictions_df[predictions_df["league"] == league]
    accuracy = league_df["prediction"] == league_df["result"]
    accuracies[league] = {"count": len(league_df), "accuracy": accuracy.mean()}

accuracies = dict(
    sorted(accuracies.items(), key=lambda item: item[1]["accuracy"], reverse=True)
)

with open(insights_path / "OutcomePrediction_league_accuracies.json", "w") as f:
    json.dump(accuracies, f)

# Specific League Predictions Analysis

In [13]:
# Specific League Analysis
analysis_league = "LEC"
games_data = team_df[["date", "gameid", "teamname", "opponentteam", "side"]]

lec_df = predictions_df[predictions_df["league"] == analysis_league][
    ["gameid", "side", "league", "prediction", "result"]
]
lec_df = games_data.merge(lec_df, on=["gameid", "side"])

lec_df["correct"] = lec_df["prediction"] == lec_df["result"]
lec_df = lec_df.sort_values(by="date")

lec_df

,date,gameid,teamname,opponentteam,side,league,prediction,result,correct
0,2022-01-14 20:10:24,ESPORTSTMNT04_2090358,Fnatic,Team BDS,Blue,LEC,1,1,True
1,2022-01-14 20:10:24,ESPORTSTMNT04_2090358,Team BDS,Fnatic,Red,LEC,0,0,True
2,2022-01-16 17:07:19,ESPORTSTMNT01_2692407,Rogue,Astralis,Blue,LEC,1,1,True
3,2022-01-16 17:07:19,ESPORTSTMNT01_2692407,Astralis,Rogue,Red,LEC,0,0,True
4,2022-01-16 18:53:12,ESPORTSTMNT01_2692441,MAD Lions KOI,G2 Esports,Blue,LEC,0,1,False
...,...,...,...,...,...,...,...,...,...
303,2024-07-26 16:53:48,LOLTMNT06_66186,Team BDS,Karmine Corp,Red,LEC,1,1,True
304,2024-07-26 18:51:01,LOLTMNT06_67177,Team BDS,Karmine Corp,Blue,LEC,1,1,True
305,2024-07-26 18:51:01,LOLTMNT06_67177,Karmine Corp,Team BDS,Red,LEC,0,0,True
306,2024-08-11 15:07:22,LOLTMNT05_73676,G2 Esports,MAD Lions KOI,Blue,LEC,1,1,True
